#### 11.1
Ex-post Attribution using test11_1_returns.csv, test11_1_weights.csv and testout11_1.csv

In [46]:
import numpy as np
import pandas as pd

In [47]:
data1 = pd.read_csv('../testfiles/data/test11_1_returns.csv').values
data1

array([[ 0.02746765, -0.02688515, -0.00922938],
       [-0.04089565,  0.06818037,  0.07299229],
       [ 0.01997821,  0.02108566,  0.01460837],
       [ 0.02568999, -0.03751938, -0.03138748],
       [-0.02989682,  0.00859192, -0.01302884],
       [-0.0543849 ,  0.0148183 ,  0.03732275],
       [ 0.00031164, -0.02041207, -0.00251772],
       [-0.02666748, -0.00563297,  0.00547393],
       [ 0.00362793, -0.02890732, -0.01932442],
       [ 0.0467719 ,  0.01572763, -0.02890823],
       [-0.00614553,  0.02739783,  0.00310455],
       [ 0.02038012,  0.02982818,  0.02554512],
       [-0.00568652,  0.03678681,  0.00919532],
       [-0.03514694, -0.01356216,  0.01833325],
       [-0.02217222, -0.02032508, -0.01446419],
       [ 0.0047044 , -0.00643055,  0.04374362],
       [-0.02725068,  0.01464341,  0.01293307],
       [-0.03524973, -0.00850412,  0.00716154],
       [-0.03859004,  0.00523091,  0.06059899],
       [ 0.01856097,  0.00093061,  0.04494699],
       [ 0.00742697, -0.00180519, -0.010

In [48]:
data2 = pd.read_csv('../testfiles/data/test11_1_weights.csv').values.squeeze()
data2

array([0.3, 0.2, 0.5])

In [49]:
row, n_stock = data1.shape

w_dynamic = np.zeros((row, n_stock))
portfolio_ret = np.zeros(row)
w_curr = data2.copy()

# return attribution
for i in range(row):
    w_dynamic[i] = w_curr
    portfolio_ret[i] = w_curr @ data1[i]
    
    w_after = w_curr * (1 + data1[i])
    w_curr = w_after / w_after.sum()

cumulative_ret = np.prod(1 + data1, axis=0) - 1
p_cumulative_ret = np.prod(1 + portfolio_ret) - 1  # portfolio: total return = return attribution

k = np.log1p(p_cumulative_ret) / (p_cumulative_ret)
carino = np.where(
    np.isclose(portfolio_ret, 0),
    1 / k,
    np.log1p(portfolio_ret) / (k * portfolio_ret)
)

return_attrib = np.sum(data1 * w_dynamic * carino[:, None], axis=0)

# risk attribution
weighted_ret = data1 * w_dynamic

X = np.column_stack([np.ones(row), portfolio_ret])  # regression of each asset's contribution on total portfolio return
beta = np.linalg.lstsq(X, weighted_ret, rcond=None)[0][1]

portfolio_sd = np.std(portfolio_ret, ddof=1)
vol_contrib = beta * portfolio_sd

In [50]:
# output
col_name = ['Value', 'x1', 'x2', 'x3', 'Portfolio']

output = pd.DataFrame([
    ['TotalReturn', *cumulative_ret, p_cumulative_ret],
    ['Return Attribution', *return_attrib, p_cumulative_ret],
    ['Vol Attribution', *vol_contrib, portfolio_sd]
], columns=col_name)

output

,Value,x1,x2,x3,Portfolio
0,TotalReturn,-0.221446,-0.016008,0.301467,0.081098
1,Return Attribution,-0.065513,-0.002220,0.148831,0.081098
2,Vol Attribution,-0.000620,0.002827,0.012580,0.014787


#### 11.2
Ex-post Attribution to Factors, using test11_2_factor_returns.csv, test11_2_stock_returns.csv, test11_2_beta.csv, test11_2_weights.csv and testout11_2.csv

In [51]:
data3 = pd.read_csv('../testfiles/data/test11_2_factor_returns.csv')
data3

,F1,F2,F3
0,-0.009729,-0.021196,-0.001513
1,0.009787,0.021889,0.010144
2,-0.001138,0.001020,0.022977
3,0.002296,-0.007547,0.014794
4,-0.005469,0.012714,0.006679
...,...,...,...
95,-0.005266,-0.004647,0.000436
96,0.016182,0.014454,0.005146
97,-0.012780,-0.000764,-0.017559
98,-0.004471,0.012612,-0.001685


In [52]:
data4 = pd.read_csv('../testfiles/data/test11_2_stock_returns.csv')
data4

,S1,S2
0,-0.018023,-0.016827
1,0.006027,0.008002
2,-0.014827,0.022379
3,0.012654,0.013625
4,0.016032,-0.004328
...,...,...
95,0.007960,-0.000021
96,0.015474,0.018301
97,0.010160,-0.013976
98,-0.000941,-0.030228


In [53]:
data5 = pd.read_csv('../testfiles/data/test11_2_beta.csv', index_col='Stock')
data5

,F1,F2,F3
Stock,,,
S1,0.638645,0.373958,-0.225069
S2,0.981400,-0.382539,0.989702


In [54]:
data6 = pd.read_csv('../testfiles/data/test11_2_weights.csv').values.squeeze()
data6

array([0.5, 0.5])

In [55]:
# stock return = beta1 * F1 + beta2 * F2 + beta3 * F3 + alpha
row, n_stock = data4.shape
n_factor = data3.shape[1]

w_dynamic = np.zeros((row, n_stock))
w_curr = data6.copy()

portfolio_ret = np.zeros(row)
factor_contrib = np.zeros((row, n_factor))

for i in range(row):
    w_dynamic[i] = w_curr
    portfolio_ret[i] = w_curr @ data4.values[i]
    
    w_after = w_curr * (1 + data4.values[i])
    w_curr = w_after / w_after.sum()

for i in range(row):
    for j, f in enumerate(data3.columns):
        factor_contrib[i, j] = np.sum(   # how much factor j contributed to portfolio return in period i
            w_dynamic[i] * data5[f].values * data3.iloc[i, j]
        )

alpha_ret = portfolio_ret - factor_contrib.sum(axis=1)

total_factor_ret = np.prod(1 + data3.values, axis=0) - 1
total_alpha_ret = np.prod(1 + alpha_ret) - 1
p_cumulative_ret = np.prod(1 + portfolio_ret) - 1  # portfolio: total return = return attribution

k = np.log1p(p_cumulative_ret) / (p_cumulative_ret)
carino = np.where(
    np.isclose(portfolio_ret, 0),
    1 / k,
    np.log1p(portfolio_ret) / (k * portfolio_ret)
)

return_attrib_factor = np.sum(factor_contrib * carino[:, None], axis=0)
return_attrib_alpha = np.sum(alpha_ret * carino)

components = np.column_stack([factor_contrib, alpha_ret])
X = np.column_stack([np.ones(row), portfolio_ret])  # regression of components on total portfolio return
beta = np.linalg.lstsq(X, components, rcond=None)[0][1]

portfolio_sd = np.std(portfolio_ret, ddof=1)
vol_contrib = beta * portfolio_sd

In [57]:
# output
col_name = ['Value', 'F1', 'F2', 'F3', 'Alpha', 'Portfolio']

output2 = pd.DataFrame([
    ['TotalReturn', *total_factor_ret, total_alpha_ret, p_cumulative_ret],
    ['Return Attribution', *return_attrib_factor, return_attrib_alpha, p_cumulative_ret],
    ['Vol Attribution', *vol_contrib, portfolio_sd]
], columns=col_name)

output2

,Value,F1,F2,F3,Alpha,Portfolio
0,TotalReturn,0.079939,0.026711,-0.016317,0.046443,0.105327
1,Return Attribution,0.065173,0.000174,-0.007579,0.047559,0.105327
2,Vol Attribution,0.005013,-0.000009,0.001785,0.005445,0.012234
